# Qwen3.6-27B + Cybersecurity-Dataset QLoRA 微调 + 智能代码审计

基于 ModelScope 的 `Qwen/Qwen3.6-27B` + `hcnote/Cybersecurity-Dataset` 做 QLoRA 4-bit 微调，
构建 **AI 智能代码安全审计器**。

## 模型信息

| 属性 | 值 |
|------|-----|
| 参数量 | 27.8B |
| 架构 | MLA (混合注意力) + MTP |
| BF16 存储 | ~56 GB |
| QLoRA 4-bit 存储 | ~14 GB |
| QLoRA 训练显存 | ~20 GB |
| 48 GB GPU | ✅ 可以跑 |

## 架构

```
Semgrep/CodeQL 粗筛 → Qwen3.6-27B (QLoRA) 二次研判 → 结构化漏洞报告
```

## 1. 安装依赖

In [ ]:
!uv pip install -U vllm modelscope transformers accelerate datasets trl peft scikit-learn pandas tqdm torchvision bitsandbytes --no-cache -i https://mirrors.cloud.tencent.com/pypi/simple/ --extra-index-url https://wheels.vllm.ai/rocm/
!pip uninstall torchaudio -y 2>/dev/null; rm -rf /opt/venv/lib/python3.12/site-packages/torchaudio /opt/venv/lib/python3.12/site-packages/torchaudio-*.dist-info

## 2. 导入与全局配置

In [ ]:
import os, glob, json, random, re, warnings
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import Dataset, DatasetDict, load_dataset
from modelscope import snapshot_download
from modelscope.hub.snapshot_download import dataset_snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── 配置 ──────────────────────────────
MODELSCOPE_MODEL_ID = "Qwen/Qwen3.6-27B"
MODELSCOPE_DATASET_ID = "hcnote/Cybersecurity-Dataset"
OUTPUT_DIR = "./qwen3.6-27b-cybersec-qlora"

TRAIN_LIMIT   = 4000
VAL_LIMIT     = 400
TEST_LIMIT    = 400
EVAL_LIMIT    = 50
MAX_INPUT_TOKENS = 768

SEED = 42
MODEL_DTYPE = torch.bfloat16
BF16, FP16 = True, False

SYSTEM_PROMPT = """You are a cybersecurity expert assistant.
Answer the user's security-related questions accurately and concisely."""

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)
os.makedirs("./datasets", exist_ok=True)

print("Model:", MODELSCOPE_MODEL_ID)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_mem / 1e9, "GB")

## 3. 固定随机种子

In [ ]:
def setup_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
setup_seed(SEED)

## 4. 下载模型

In [ ]:
print("Downloading", MODELSCOPE_MODEL_ID)
model_dir = snapshot_download(MODELSCOPE_MODEL_ID, cache_dir="./models")
LOCAL_MODEL_DIR = model_dir
print("Model dir:", LOCAL_MODEL_DIR)

## 5. 加载 tokenizer（先于数据集，用于精准过滤）

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR, use_fast=True, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if not getattr(tokenizer, "chat_template", None):
    tmpl_dir = snapshot_download(MODELSCOPE_MODEL_ID, cache_dir="./models",
                                  allow_file_pattern=["chat_template.jinja"])
    tmpl_path = os.path.join(tmpl_dir, "chat_template.jinja")
    if os.path.exists(tmpl_path):
        with open(tmpl_path) as f:
            tokenizer.chat_template = f.read()

_probe = tokenizer.apply_chat_template(
    [{"role":"system","content":"Hi"},{"role":"user","content":"Hello"}],
    tokenize=False, add_generation_prompt=True)
print("Chat template probe:\n", _probe[:200])
print("pad:", tokenizer.pad_token, "| bos:", tokenizer.bos_token, "| eos:", tokenizer.eos_token)

## 6. 加载数据集（tokenizer 精准过滤）

In [ ]:
dataset_dir = dataset_snapshot_download(MODELSCOPE_DATASET_ID, cache_dir="./datasets")
jsonl_files = sorted(glob.glob(os.path.join(dataset_dir, "*.jsonl")))
print(f"JSONL files: {jsonl_files}")

raw = load_dataset("json", data_files=jsonl_files, split="train")
print(f"Raw: {len(raw)} samples")

def not_too_long(ex):
    text = ex.get("instruction","") + "\n\n" + ex.get("input","")
    return len(tokenizer.encode(text, truncation=False)) <= MAX_INPUT_TOKENS

raw = raw.filter(not_too_long, desc="Filtering long samples")
print(f"After filter: {len(raw)} samples (<= {MAX_INPUT_TOKENS} tokens)")

raw = raw.shuffle(seed=SEED)
total = len(raw)
t_end = min(TRAIN_LIMIT, total)
v_end = t_end + min(VAL_LIMIT, max(0, total - t_end))
s_end = v_end + min(TEST_LIMIT, max(0, total - v_end))

dataset = DatasetDict({
    "train": raw.select(range(0, t_end)),
    "validation": raw.select(range(t_end, v_end)),
    "test": raw.select(range(v_end, s_end)),
})
print(dataset)

## 7. 构造 SFT 数据

In [ ]:
def to_prompt_completion(ex):
    instr = ex.get("instruction", "")
    inp = ex.get("input", "")
    out = ex.get("output", "")
    user_content = f"{instr}\n\n{inp}" if inp else instr
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "completion": [{"role": "assistant", "content": out}],
    }

sft_dataset = dataset.map(to_prompt_completion,
    remove_columns=dataset["train"].column_names)
print(sft_dataset)
print(sft_dataset["train"][0])

## 8. 加载模型（QLoRA 4-bit）

Qwen3.6-27B BF16 全量 ~56 GB，48 GB 放不下。
使用 4-bit 量化加载（~14 GB），加上训练开销 ~20 GB。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# QLoRA 4-bit 量化加载
print("Loading with QLoRA 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=MODEL_DTYPE,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    quantization_config=bnb_config,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map="auto",
)

base_model = prepare_model_for_kbit_training(base_model)

base_model.config.use_cache = False
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.bos_token_id = tokenizer.bos_token_id
base_model.config.eos_token_id = tokenizer.eos_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.bos_token_id = tokenizer.bos_token_id
base_model.generation_config.eos_token_id = tokenizer.eos_token_id

print("Model loaded.")
print("VRAM used:", torch.cuda.memory_allocated() / 1e9, "GB")

## 9. 推理函数

In [ ]:
def generate_response(model, tok, instruction, user_input="",
                      system=SYSTEM_PROMPT, max_new=256):
    content = f"{instruction}\n\n{user_input}" if user_input else instruction
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": content},
    ]
    dev = next(model.parameters()).device
    inputs = tok.apply_chat_template(msgs, tokenize=True,
        add_generation_prompt=True, return_dict=True, return_tensors="pt")
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    ilen = inputs["input_ids"].shape[-1]
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
            pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(out[0][ilen:], skip_special_tokens=True).strip()

print(generate_response(base_model, tokenizer,
    "What is SQL injection?", max_new=128)[:300])

## 10. 评估函数

In [ ]:
def evaluate_model(model, tok, split="test", limit=EVAL_LIMIT):
    rows = []
    src = dataset[split]
    if limit: src = src.select(range(min(limit, len(src))))
    model.eval()
    for ex in tqdm(src, desc=f"Eval {split}", leave=False):
        pred = generate_response(model, tok,
            ex.get("instruction",""), ex.get("input",""), max_new=256)
        rows.append({"instruction": ex.get("instruction",""),
            "input": ex.get("input",""),
            "reference": ex.get("output",""), "prediction": pred})
    return pd.DataFrame(rows)

def print_comparison(pre_df, post_df, n=5):
    for i in range(min(n, len(pre_df))):
        print(f"{'='*60}\nSample {i+1}")
        print(f"Q: {pre_df.iloc[i]['instruction'][:150]}")
        print(f"Ref: {pre_df.iloc[i]['reference'][:200]}")
        print(f"Pre: {pre_df.iloc[i]['prediction'][:200]}")
        print(f"Post: {post_df.iloc[i]['prediction'][:200]}")

## 11. 微调前评估

In [ ]:
pre_preds = evaluate_model(base_model, tokenizer, limit=EVAL_LIMIT)
print(f"Pre-finetune: {len(pre_preds)} samples")
pre_preds[["instruction","reference","prediction"]].head(3)

## 12. LoRA 配置

QLoRA 模式下 `target_modules="all-linear"` 自动适配量化层。

In [ ]:
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules="all-linear",
)

## 13. 训练参数

QLoRA 模式下 `bf16=True` 自动转为 compute dtype（`bnb_4bit_compute_dtype`）。

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4, weight_decay=0.01,
    lr_scheduler_type="linear", warmup_steps=50, num_train_epochs=1,
    logging_steps=5, eval_strategy="steps", eval_steps=25,
    save_strategy="steps", save_steps=25, save_total_limit=2,
    metric_for_best_model="eval_loss", greater_is_better=False,
    gradient_checkpointing=True,
    bf16=BF16, fp16=FP16, tf32=False,
    max_length=512, packing=False, completion_only_loss=True,
    remove_unused_columns=False, dataloader_num_workers=2,
    optim="adamw_torch",
    report_to="none",
    seed=SEED, data_seed=SEED,
)

## 14. LoRA / QLoRA 微调

In [ ]:
if isinstance(base_model, PeftModel):
    base_model = base_model.unload()
    base_model.config.use_cache = False

trainer = SFTTrainer(
    model=base_model, train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    peft_config=lora_config, args=training_args, processing_class=tokenizer,
)

trainable = 0; total_p = 0
for n, p in trainer.model.named_parameters():
    total_p += p.numel()
    if p.requires_grad: trainable += p.numel()

print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")
if trainable == 0: raise RuntimeError("No LoRA params!")

train_result = trainer.train()
trainer.model.eval()
trainer.model.config.use_cache = True
train_result

## 15. 保存 Adapter

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(f"{OUTPUT_DIR}/train_metrics.json","w") as f:
    json.dump(train_result.metrics, f, indent=2)
print("Saved to", OUTPUT_DIR)

## 16. 微调后评估

In [ ]:
ft_model = trainer.model
ft_model.eval()
ft_model.config.use_cache = True
post_preds = evaluate_model(ft_model, tokenizer, limit=EVAL_LIMIT)
print(f"Post-finetune: {len(post_preds)} samples")
print_comparison(pre_preds, post_preds, n=3)

## 17. 智能代码审计面板

复用已加载的 `ft_model`，零额外显存。

支持：代码片段分析、Semgrep 二次研判、文件审计。

In [ ]:
AUDIT_SYSTEM = """You are a senior application security engineer performing code review.
Analyze the code for security vulnerabilities.

For each vulnerability found, output EXACTLY in this format:

[VULN] <CWE-ID> <vulnerability type>
[SEVERITY] critical|high|medium|low
[LINE] <line number or range>
[DESC] <brief description>
[FIX] <concrete fix recommendation with code example>

If no vulnerabilities found, output: [SAFE] No security vulnerabilities detected.

Be thorough. Check for: injection, XSS, broken auth, sensitive data exposure,
XXE, broken access control, misconfiguration, insecure deserialization,
using components with known vulnerabilities, insufficient logging."""


class CodeAuditor:
    """AI 智能代码安全审计器"""

    def __init__(self, model, tok):
        self.model = model
        self.tok = tok
        self.dev = next(model.parameters()).device

    def audit(self, code: str, language: str = "", context: str = "") -> str:
        instruction = f"Audit this {language} code for security vulnerabilities."
        user_input = f"```{language}\n{code}\n```"
        if context:
            user_input += f"\n\nStatic analysis context:\n{context}"

        msgs = [
            {"role": "system", "content": AUDIT_SYSTEM},
            {"role": "user", "content": f"{instruction}\n\n{user_input}"},
        ]
        inputs = self.tok.apply_chat_template(msgs, tokenize=True,
            add_generation_prompt=True, return_dict=True, return_tensors="pt")
        inputs = {k: v.to(self.dev) for k, v in inputs.items()}
        ilen = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=512,
                do_sample=False, pad_token_id=self.tok.pad_token_id,
                eos_token_id=self.tok.eos_token_id)
        return self.tok.decode(out[0][ilen:], skip_special_tokens=True).strip()

    def audit_file(self, path: str) -> str:
        with open(path) as f:
            code = f.read()
        ext = path.rsplit(".",1)[-1] if "." in path else ""
        lang = {"py":"Python","c":"C","cpp":"C++","java":"Java",
                "go":"Go","js":"JavaScript","ts":"TypeScript",
                "php":"PHP","rb":"Ruby","rs":"Rust","swift":"Swift",
                "sh":"Bash","sql":"SQL","yaml":"YAML","yml":"YAML",
                "tf":"Terraform","dockerfile":"Dockerfile"}.get(ext.lower(), ext)
        return self.audit(code, lang)

    def audit_with_semgrep(self, code: str, language: str,
                           semgrep_finding: str) -> str:
        """Semgrep 二次研判 — 降误报"""
        context = (
            f"Static analysis (Semgrep) flagged this:\n{semgrep_finding}\n\n"
            "Verify if TRUE positive or FALSE positive. Explain reasoning."
        )
        return self.audit(code, language, context)

    def parse_report(self, raw: str) -> list[dict]:
        findings = []
        current = {}
        for line in raw.split("\n"):
            line = line.strip()
            if line.startswith("[VULN]"):
                if current:
                    findings.append(current)
                parts = line[6:].strip().split(None, 1)
                current = {"cwe": parts[0] if parts else "?",
                           "type": parts[1] if len(parts)>1 else line[6:].strip()}
            elif line.startswith("[SEVERITY]"):
                current["severity"] = line[10:].strip()
            elif line.startswith("[LINE]"):
                current["line"] = line[6:].strip()
            elif line.startswith("[DESC]"):
                current["description"] = line[6:].strip()
            elif line.startswith("[FIX]"):
                current["fix"] = line[5:].strip()
            elif "[SAFE]" in line:
                return []
        if current:
            findings.append(current)
        return findings

    def audit_and_report(self, code: str, language: str = "",
                         context: str = "") -> pd.DataFrame:
        raw = self.audit(code, language, context)
        findings = self.parse_report(raw)
        if not findings:
            print("\u2705 No vulnerabilities found.")
            return pd.DataFrame()
        df = pd.DataFrame(findings)
        print(f"\U0001f50d Found {len(df)} potential issue(s):\n")
        for _, row in df.iterrows():
            icon = {"critical":"\U0001f534","high":"\U0001f7e0",
                    "medium":"\U0001f7e1","low":"\U0001f7e2"}.get(
                row.get("severity",""), "\u26aa")
            print(f"{icon} [{row.get('severity','?').upper()}] {row.get('type','?')}")
            print(f"   CWE: {row.get('cwe','?')} | Line: {row.get('line','?')}")
            print(f"   {row.get('description','')[:120]}")
            if row.get('fix'):
                print(f"   Fix: {row.get('fix','')[:150]}")
            print()
        return df


auditor = CodeAuditor(ft_model, tokenizer)
print("CodeAuditor initialized.")
print("  auditor.audit_and_report(code, 'python')")
print("  auditor.audit_file('app.py')")
print("  auditor.audit_with_semgrep(code, 'python', finding)")

## 18. 审计测试

In [ ]:
# SQL 注入
auditor.audit_and_report(
    'def get_user(uid):\n    q = "SELECT * FROM users WHERE id=" + uid\n    return db.execute(q)',
    "python")

# 命令注入
auditor.audit_and_report(
    'import os\ndef ping(h):\n    os.system("ping -c1 " + h)',
    "python")

# 硬编码密钥
auditor.audit_and_report(
    'API_KEY = "sk-1234567890abcdef"\ndef call():\n    return requests.get("https://api.x.com", headers={"Authorization": f"Bearer {API_KEY}"})',
    "python")

## 19. Semgrep 集成（可选）

In [ ]:
def triage_semgrep(semgrep_json_path: str, source_dir: str):
    with open(semgrep_json_path) as f:
        data = json.load(f)
    results = []
    for finding in data.get("results", []):
        rule = finding.get("check_id", "?")
        path = finding.get("path", "")
        line = finding.get("start", {}).get("line", "?")
        msg = finding.get("extra", {}).get("message", "")
        sev = finding.get("extra", {}).get("severity", "info")

        try:
            with open(os.path.join(source_dir, path)) as fh:
                lines = fh.readlines()
            code = "".join(lines[max(0,line-6):min(len(lines),line+5)])
        except Exception:
            code = "[unreadable]"

        verdict = auditor.audit_with_semgrep(
            code, "", f"Rule: {rule}\nSeverity: {sev}\nMessage: {msg}")
        results.append({"file": path, "line": line, "rule": rule,
                        "semgrep_sev": sev, "llm_verdict": verdict})
        print(f"  [{rule}] {path}:{line}")
    return pd.DataFrame(results)

print("Usage: df = triage_semgrep('semgrep.json', 'src/')")

## 20. 保存结果

In [ ]:
pre_preds.to_csv(f"{OUTPUT_DIR}/pre_finetuning.csv", index=False)
post_preds.to_csv(f"{OUTPUT_DIR}/post_finetuning.csv", index=False)
comp = pre_preds[["instruction","input","reference"]].copy()
comp["pre"] = pre_preds["prediction"]
comp["post"] = post_preds["prediction"]
comp.to_csv(f"{OUTPUT_DIR}/comparison.csv", index=False)
print("Saved to", OUTPUT_DIR)

## 常见问题

### QLoRA 在 ROCm 上不兼容怎么办

确认 `bitsandbytes` 版本 ≥ 0.43.0，ROCm ≥ 5.6。如果仍不兼容，可尝试 `pip install bitsandbytes --upgrade`。

### OOM 怎么办

1. 降低 `MAX_INPUT_TOKENS` (512)
2. 降低 `per_device_train_batch_size` (1 或 2)
3. 降低 `max_length` (384)
4. 降低 `TRAIN_LIMIT`

### 想换模型怎么办

修改 Cell 2 的 `MODELSCOPE_MODEL_ID`，Qwen3.6-27B 必须配合 QLoRA（`load_in_4bit=True`），如果换 9B 可以改为 BF16 全量加载。